## **Aim**
To implement a program that identifies possible password attacks by analyzing repeated authentication failures.

## **Algorithm**
**Step 1:** Import `re`, `collections.Counter`, `datetime`, and `json` libraries.

**Step 2:** Create a simulated authentication log with fields: timestamp, username, source_ip, result (success/failure), method (password/ssh_key/mfa).

**Step 3:** Parse the log and extract failed authentication attempts.

**Step 4:** Group failures by:
   - Source IP (brute force from single IP)
   - Username (credential stuffing)
   - Username + IP (targeted attack)

**Step 5:** Detect attack patterns:
   - Brute force: Many failures for one username from one IP
   - Credential stuffing: One username tried from many IPs
   - Password spraying: One password tried against many usernames
   - Distributed attacks: Same pattern from multiple IPs

**Step 6:** Calculate time windows and rates.

**Step 7:** Generate attack report with severity classification.

In [1]:
import re
import json
from collections import Counter, defaultdict
from datetime import datetime, timedelta

def create_sample_auth_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=24)
    
    events = []
    
    # Normal successful logins
    for i in range(50):
        events.append({
            "timestamp": (base + timedelta(minutes=i*20)).isoformat(),
            "username": "user",
            "source_ip": "192.168.1.50",
            "result": "success",
            "method": "password"
        })
    
    for i in range(30):
        events.append({
            "timestamp": (base + timedelta(minutes=i*30)).isoformat(),
            "username": "admin",
            "source_ip": "192.168.1.200",
            "result": "success",
            "method": "ssh_key"
        })
    
    # Brute force from 10.0.0.100 - targeting administrator
    for i in range(25):
        events.append({
            "timestamp": (base + timedelta(hours=2, minutes=i)).isoformat(),
            "username": "administrator",
            "source_ip": "10.0.0.100",
            "result": "failure",
            "method": "password"
        })
    
    for i in range(15):
        events.append({
            "timestamp": (base + timedelta(hours=2, minutes=30+i)).isoformat(),
            "username": "admin",
            "source_ip": "10.0.0.100",
            "result": "failure",
            "method": "password"
        })
    
    for i in range(10):
        events.append({
            "timestamp": (base + timedelta(hours=3, minutes=i*3)).isoformat(),
            "username": "root",
            "source_ip": "10.0.0.100",
            "result": "failure",
            "method": "password"
        })
    
    # Credential stuffing - john from multiple IPs
    stuffing_ips = ["203.0.113.10", "203.0.113.20", "203.0.113.30", "203.0.113.40",
                  "198.51.100.10", "198.51.100.20", "198.51.100.30", "192.0.2.10"]
    for ip in stuffing_ips:
        events.append({
            "timestamp": (base + timedelta(hours=4)).isoformat(),
            "username": "john",
            "source_ip": ip,
            "result": "failure",
            "method": "password"
        })
    
    # Credential stuffing - jane from multiple IPs
    for ip in stuffing_ips[:7]:
        events.append({
            "timestamp": (base + timedelta(hours=5)).isoformat(),
            "username": "jane",
            "source_ip": ip,
            "result": "failure",
            "method": "password"
        })
    
    # Password spraying from 172.16.0.50
    spray_users = ["user1", "user2", "user3", "user4", "user5", "user6", "user7", "user8",
                 "user9", "user10", "user11", "user12", "user13", "user14", "user15", "user16",
                 "user17", "user18", "user19", "user20"]
    for user in spray_users:
        events.append({
            "timestamp": (base + timedelta(hours=6, minutes=spray_users.index(user)*2)).isoformat(),
            "username": user,
            "source_ip": "172.16.0.50",
            "result": "failure",
            "method": "password"
        })
    
    # Root brute force from Tor
    for i in range(12):
        events.append({
            "timestamp": (base + timedelta(hours=8, minutes=i*2)).isoformat(),
            "username": "root",
            "source_ip": "203.0.113.45",
            "result": "failure",
            "method": "password"
        })
    
    # Some other noise
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(hours=10, minutes=i*10)).isoformat(),
            "username": "test",
            "source_ip": "192.168.1.50",
            "result": "failure",
            "method": "password"
        })
    
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(hours=11, minutes=i*10)).isoformat(),
            "username": "user",
            "source_ip": "192.168.1.50",
            "result": "failure",
            "method": "password"
        })
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_password_attacks(log_file):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    failures = [e for e in events if e["result"] == "failure"]
    
    # Group by username + IP
    user_ip_failures = defaultdict(list)
    for f in failures:
        key = (f["username"], f["source_ip"])
        user_ip_failures[key].append(f)
    
    # Group by username only (credential stuffing)
    user_failures = defaultdict(list)
    for f in failures:
        user_failures[f["username"]].append(f)
    
    # Group by IP only (password spraying)
    ip_failures = defaultdict(list)
    for f in failures:
        ip_failures[f["source_ip"]].append(f)
    
    attacks = []
    
    # Detect brute force (many failures for same user from same IP in short time)
    for (user, ip), attempts in user_ip_failures.items():
        if len(attempts) >= 5:
            attempts.sort(key=lambda x: x["timestamp"])
            first = datetime.fromisoformat(attempts[0]["timestamp"])
            last = datetime.fromisoformat(attempts[-1]["timestamp"])
            duration_min = (last - first).total_seconds() / 60
            if duration_min == 0:
                duration_min = 1
            rate_per_hr = len(attempts) / (duration_min / 60)
            
            if rate_per_hr >= 30 and len(attempts) >= 10:
                severity = "CRITICAL"
            elif rate_per_hr >= 15 and len(attempts) >= 5:
                severity = "HIGH"
            else:
                severity = "MEDIUM"
            
            attacks.append({
                "type": "BRUTE_FORCE",
                "target": user, "source": ip,
                "failures": len(attempts), "duration_min": round(duration_min, 1),
                "rate_per_hr": round(rate_per_hr), "methods": [attempts[0]["method"]],
                "severity": severity
            })
    
    # Detect credential stuffing (same username from many IPs)
    for user, attempts in user_failures.items():
        unique_ips = set(a["source_ip"] for a in attempts)
        if len(unique_ips) >= 5 and len(attempts) >= 5:
            attacks.append({
                "type": "CREDENTIAL_STUFFING",
                "target": user,
                "sources": list(unique_ips),
                "failures": len(attempts),
                "pattern": "Single username from multiple IPs",
                "severity": "HIGH"
            })
    
    # Detect password spraying (one IP, many usernames)
    for ip, attempts in ip_failures.items():
        unique_users = set(a["username"] for a in attempts)
        if len(unique_users) >= 10 and len(attempts) >= 10:
            attacks.append({
                "type": "PASSWORD_SPRAYING",
                "source": ip,
                "targets": list(unique_users),
                "failures": len(attempts),
                "pattern": "One password tried against many users",
                "severity": "HIGH"
            })
    
    return attacks, failures, user_failures, ip_failures

def main():
    log_file = "auth_attack_log.json"
    create_sample_auth_log(log_file)
    
    print("Analyzing authentication logs for password attacks...")
    attacks, failures, user_failures, ip_failures = analyze_password_attacks(log_file)
    
    print(f"\n{'='*60}")
    print(f"PASSWORD ATTACK DETECTION REPORT")
    print(f"{'='*60}")
    print(f"Total auth attempts: {len([e for e in json.load(open(log_file)) if e['result'] in ('success', 'failure')])}")
    print(f"Failed attempts: {len(failures)}")
    print(f"Unique source IPs: {len(ip_failures)}")
    print(f"Unique usernames: {len(user_failures)}")
    print(f"Time window: 24 hours")
    
    print(f"\n--- ATTACK PATTERNS DETECTED ---")
    
    for i, a in enumerate(attacks, 1):
        print(f"\n{i}. [{a['severity']}] {a['type']}")
        if a["type"] == "BRUTE_FORCE":
            print(f"   Target: {a['target']}")
            print(f"   Source: {a['source']}")
            print(f"   Failures: {a['failures']} in {a['duration_min']} min")
            print(f"   Rate: {a['rate_per_hr']}/hr")
            print(f"   Methods: {', '.join(a['methods'])}")
        elif a["type"] == "CREDENTIAL_STUFFING":
            print(f"   Target: {a['target']}")
            print(f"   Sources: {len(a['sources'])} IPs")
            print(f"   Failures: {a['failures']} total")
            print(f"   Pattern: {a['pattern']}")
        elif a["type"] == "PASSWORD_SPRAYING":
            print(f"   Source: {a['source']}")
            print(f"   Targets: {len(a['targets'])} usernames")
            print(f"   Failures: {a['failures']} (1 per user)")
            print(f"   Pattern: {a['pattern']}")
    
    # Top attacked usernames
    print(f"\n--- TOP ATTACKED USERNAMES ---")
    user_counts = Counter(len(v) for v in user_failures.values())
    # Actually let's do it properly
    user_totals = {u: len(v) for u, v in user_failures.items()}
    for i, (user, count) in enumerate(sorted(user_totals.items(), key=lambda x: -x[1])[:10], 1):
        print(f"{i}. {user}: {count} failures")
    
    # Top attacker IPs
    print(f"\n--- TOP ATTACKER IPs ---")
    ip_totals = {ip: len(v) for ip, v in ip_failures.items()}
    for i, (ip, count) in enumerate(sorted(ip_totals.items(), key=lambda x: -x[1])[:10], 1):
        print(f"{i}. {ip}: {count} failures")
    
    # Summary
    severity_counts = Counter(a["severity"] for a in attacks)
    print(f"\n--- SUMMARY ---")
    for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        if sev in severity_counts:
            print(f"  {sev}: {severity_counts[sev]}")

if __name__ == "__main__":
    main()

Analyzing authentication logs for password attacks...

PASSWORD ATTACK DETECTION REPORT
Total auth attempts: 187
Failed attempts: 107
Unique source IPs: 12
Unique usernames: 27
Time window: 24 hours

--- ATTACK PATTERNS DETECTED ---

1. [CRITICAL] BRUTE_FORCE
   Target: administrator
   Source: 10.0.0.100
   Failures: 25 in 24.0 min
   Rate: 62/hr
   Methods: password

2. [CRITICAL] BRUTE_FORCE
   Target: admin
   Source: 10.0.0.100
   Failures: 15 in 14.0 min
   Rate: 64/hr
   Methods: password

3. [HIGH] BRUTE_FORCE
   Target: root
   Source: 10.0.0.100
   Failures: 10 in 27.0 min
   Rate: 22/hr
   Methods: password

4. [CRITICAL] BRUTE_FORCE
   Target: root
   Source: 203.0.113.45
   Failures: 12 in 22.0 min
   Rate: 33/hr
   Methods: password

5. [MEDIUM] BRUTE_FORCE
   Target: test
   Source: 192.168.1.50
   Failures: 5 in 40.0 min
   Rate: 8/hr
   Methods: password

6. [MEDIUM] BRUTE_FORCE
   Target: user
   Source: 192.168.1.50
   Failures: 5 in 40.0 min
   Rate: 8/hr
   Methods

## **Result**
This the program successfully identifies possible password attacks by analyzing repeated authentication failures.